# Chapter 3

## Summarizing a document bigger than the LLM’s context window

In [1]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [3]:
import os

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter

# Reads OPENAI_API_KEY / OPENAI_BASE_URL / OPENAI_MODEL from .env
load_dotenv(override=True)

True

In [4]:
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")

In [5]:
llm = ChatOpenAI(model=OPENAI_MODEL)

In [10]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x:
        [
            {
                'chunk': text_chunk,
            }
            for text_chunk in
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [11]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

In [12]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x:
        {
            'summaries': '\n'.join([i['summary'] for i in x]),
        })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

In [13]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)

In [14]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [15]:
print(summary)

The text is the Project Gutenberg edition of Melville’s *Moby-Dick* (eBook #2701), containing the license notice and the opening chapters.

- **“Loomings”:** Ishmael goes to sea whenever his soul becomes “a damp, drizzly November.” He describes the ocean’s mysterious pull—linked to Narcissus and “the ungraspable phantom of life”—and insists on sailing as a common sailor, not an officer. He accepts that “who ain’t a slave?”, values pay and fresh air, and chooses a whaling voyage as a providentially fated part of his life, drawn by the whale and “an everlasting itch for things remote.”
- **Chapter 2:** Arriving in New Bedford on a cold December Saturday, he learns the Nantucket packet has already sailed. Searching for cheap lodging, he rejects the Crossed Harpoons and Sword-Fish Inn, briefly enters a church called “The Trap,” and settles at the Spouter-Inn, kept by Peter Coffin.
- **Chapter 3:** He puzzles over a defaced painting, finally reading it as a Cape Horner in a hurricane with a

## Summarizing across documents

In [16]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [20]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [21]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [25]:
llm = ChatOpenAI(model=OPENAI_MODEL)

In [26]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [27]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [28]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary,
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)

        current_refined_summary = refine_chain.invoke(intermediate_step)

    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [31]:
full_summary = refine_summary(all_docs)
print(full_summary)

{'final_summary': '## Final Summary: Paestum\n\n**Overview**\nPaestum (PEST-əm; Latin: [ˈpae̯stũː]), Greek **Poseidonia**, was a major ancient Greek city in southern Italy near the west coast, about 22 miles (35 km) southeast of modern Salerno and 5 miles (8 km) south of the Sele (ancient Silarus) River. A major city of Magna Graecia on the Tyrrhenian coast, it is celebrated for three exceptionally well-preserved Doric temples dating from roughly 550–450 BCE (a narrower range of c. 530–c. 460 BC is also given for the surviving Doric group). The city walls and amphitheatre are largely intact, along with paved roads and the lower walls of many structures. The site is open to the public and includes a modern national museum housing finds from Paestum as well as from the associated Greek site of Foce del Sele.\n\n**History**\n- **Founding:** Established around 600 BCE by Greek settlers from Sybaris, a Greek colony in southern Italy, as **Poseidonia** (Ποσειδωνία). Strabo, like Aristotle, h